# FHIR R4 Terminology Server Demo

This notebook demonstrates the medterm4ds FHIR R4 terminology server — a
UMLS-backed terminology service supporting code lookup, validation,
cross-system mapping, hierarchy checking, ValueSet expansion, and
text-to-code search (lexical + semantic).

**Prerequisites**: `pip install medterm4ds[fhir]` and a UMLS DuckDB at
`MEDTERM4DS_DB`.

## Start the server

In [ ]:
import os
os.environ.setdefault('MEDTERM4DS_DB', '/mnt/d/medterm4ds/data/umls_current.duckdb')
os.environ.setdefault('MEDTERM4DS_FHIR4PX_BASELINE', '/mnt/d/medterm4ds/reports/fhir4px')
os.environ.setdefault('MEDTERM4DS_SEARCH_INDEX_DIR', '/mnt/d/fhir4px-model/dist/naming_bm25')

from starlette.testclient import TestClient
from medterm4ds.apps.fhir_api import FhirApiSettings, create_fhir_app

settings = FhirApiSettings.from_env()
app = create_fhir_app(settings)
client = TestClient(app)

# Warm up the lifespan
with client:
    resp = client.get('/fhir/metadata')
    print(f'Server status: {resp.status_code}')
    print(f'FHIR version: {resp.json()["fhirVersion"]}')
    ops = [op['name'] for r in resp.json()['rest'] for res in r['resource'] for op in res.get('operation', [])]
    print(f'Operations: {sorted(ops)}')

---
## 1. $lookup — Code Details

Given a code, return its display name and custom properties (patient-friendly
name, canonical ICD-10 code, term type).

In [ ]:
import json

def show_params(resp, title=''):
    """Pretty-print a FHIR Parameters response."""
    body = resp.json()
    if title:
        print(f'\n=== {title} ===')
    print(f'Status: {resp.status_code} | Resource: {body["resourceType"]}')
    for p in body.get('parameter', []):
        name = p['name']
        if 'part' in p:
            # Property entry: code + value
            parts = {pt['name']: pt.get('valueString', pt.get('valueCode', '')) for pt in p['part']}
            print(f'  {name}: {parts.get("code", "?")} = {parts.get("value", "?")}')
        else:
            val = p.get('valueString', p.get('valueUri', p.get('valueCode', p.get('valueBoolean', '?'))))
            print(f'  {name}: {val}')

# Lookup a SNOMED code
resp = client.get('/fhir/CodeSystem/$lookup', params={
    'system': 'http://snomed.info/sct',
    'code': '44054006',
})
show_params(resp, 'SNOMED 44054006 (Type 2 diabetes)')

In [ ]:
# Lookup a RxNorm code (shows tty property)
resp = client.get('/fhir/CodeSystem/$lookup', params={
    'system': 'http://www.nlm.nih.gov/research/umls/rxnorm',
    'code': '860975',
})
show_params(resp, 'RxNorm 860975 (Metformin)')

---
## 2. $validate-code — Code Validation

Check if a code exists in a code system.

In [ ]:
# Valid code
resp = client.get('/fhir/CodeSystem/$validate-code', params={
    'system': 'http://snomed.info/sct', 'code': '44054006',
})
result = [p for p in resp.json()['parameter'] if p['name'] == 'result'][0]
print(f'44054006 valid: {result["valueBoolean"]}')  # True

# Invalid code
resp = client.get('/fhir/CodeSystem/$validate-code', params={
    'system': 'http://snomed.info/sct', 'code': 'FAKE999',
})
result = [p for p in resp.json()['parameter'] if p['name'] == 'result'][0]
print(f'FAKE999 valid: {result["valueBoolean"]}')  # False

---
## 3. $translate — Cross-System Mapping

Map a code from one system to another (e.g., SNOMED → ICD-10).

In [ ]:
resp = client.get('/fhir/ConceptMap/$translate', params={
    'system': 'http://snomed.info/sct',
    'code': '44054006',
    'targetsystem': 'http://hl7.org/fhir/sid/icd-10-cm',
})
body = resp.json()
result = [p for p in body['parameter'] if p['name'] == 'result'][0]
print(f'Translate result: {result["valueBoolean"]}')
for p in body['parameter']:
    if p['name'] == 'match':
        parts = {pt['name']: pt for pt in p['part']}
        concept = parts.get('concept', {}).get('valueCoding', {})
        equiv = parts.get('equivalence', {}).get('valueCode', '?')
        print(f'  → {concept.get("code", "?")} ({concept.get("display", "?")}) [{equiv}]')

---
## 4. $subsumes — Hierarchy Checking

Check if one code is an ancestor of another (subsumption).

In [ ]:
cases = [
    ('73211009', '44054006', 'Diabetes → Type 2 diabetes'),
    ('44054006', '73211009', 'Type 2 diabetes → Diabetes'),
    ('44054006', '44054006', 'Same code'),
    ('44054006', '860975',  'Diabetes → Metformin'),
]
for code_a, code_b, desc in cases:
    resp = client.get('/fhir/CodeSystem/$subsumes', params={
        'system': 'http://snomed.info/sct',
        'codeA': code_a,
        'codeB': code_b,
    })
    outcome = [p for p in resp.json()['parameter'] if p['name'] == 'outcome'][0]
    print(f'{desc:45s} → {outcome["valueCode"]}')

---
## 5. $expand — ValueSet Expansion

Expand a ValueSet by text filter (EHR autocomplete) or intensional definition
(all descendants of X).

In [ ]:
# Text filter: "diabetes"
resp = client.get('/fhir/ValueSet/$expand', params={'filter': 'diabetes', 'count': 5})
body = resp.json()
contains = body['expansion']['contains']
print(f'Filter "diabetes": {body["expansion"]["total"]} matches')
for c in contains:
    print(f'  {c["system"].split("/")[-1]:15s} {c["code"]:15s} {c.get("display", "")}')

In [ ]:
# Intensional: all descendants of SNOMED 73211009 (Diabetes mellitus)
resp = client.post('/fhir/ValueSet/$expand', json={
    'resourceType': 'ValueSet',
    'compose': {
        'include': [{
            'system': 'http://snomed.info/sct',
            'filter': [{'property': 'concept', 'op': 'is-a', 'value': '73211009'}],
        }],
    },
})
body = resp.json()
print(f'Intensional is-a 73211009: {body["expansion"]["total"]} codes')
for c in body['expansion']['contains'][:5]:
    print(f'  {c["code"]:15s} {c.get("display", "")}')
if body['expansion']['total'] > 5:
    print(f'  ... and {body["expansion"]["total"] - 5} more')

---
## 6. $search — Text-to-Code Search

Three modes:
- **lexical**: BM25 token matching (~1ms, covers 80-90% of queries)
- **semantic**: SapBERT embedding + FAISS ANN (~100ms, catches novel phrasings)
- **hybrid**: BM25 retrieve + SapBERT re-rank (~110ms, best accuracy)

Each result has a match-grade: `certain`, `probable`, or `possible`.

In [ ]:
def show_search(query, mode='lexical', system=None):
    params = {'query': query, 'count': 5, 'searchMode': mode}
    if system:
        params['system'] = system
    resp = client.get('/fhir/CodeSystem/$search', params=params)
    if resp.status_code != 200:
        print(f'  [{resp.status_code}] {resp.json().get("issue", [{}])[0].get("diagnostics", "")}')
        return
    body = resp.json()
    print(f'  Query: "{query}" (mode={mode}) → {body["total"]} results')
    for entry in body['entry']:
        r = entry['resource']
        score = entry['search']['score']
        grade = entry['search']['extension'][0]['valueCode']
        src = r['system'].split('/')[-1]
        print(f'    {score:.2f} {grade:8s} {src:15s} {r["code"]:15s} {r.get("display", "")}')

# Lexical search
show_search('diabetes', mode='lexical')

In [ ]:
# Semantic search: catches novel phrasings where BM25 fails
show_search('high blood sugar', mode='semantic')
show_search('chest feels tight', mode='semantic')
show_search('water pill', mode='semantic')

In [ ]:
# Hybrid: best of both worlds
show_search('metformin pill', mode='hybrid')

---
## 7. $closure — Fast Subsumption Table

Pre-compute hierarchy relationships for O(1) subsumption checks.
Useful for validating codes in large ValueSets or patient records.

In [ ]:
# Initialize a closure
resp = client.post('/fhir/CodeSystem/$closure', json={
    'resourceType': 'Parameters',
    'parameter': [{'name': 'name', 'valueString': 'demo-closure'}],
})
version = [p for p in resp.json()['parameter'] if p['name'] == 'return'][0]
print(f'Initialized closure: version={version["valueString"]}')

# Add concepts
resp = client.post('/fhir/CodeSystem/$closure', json={
    'resourceType': 'Parameters',
    'parameter': [
        {'name': 'name', 'valueString': 'demo-closure'},
        {'name': 'concept', 'valueCoding': {
            'system': 'http://snomed.info/sct', 'code': '73211009', 'display': 'Diabetes'}},
        {'name': 'concept', 'valueCoding': {
            'system': 'http://snomed.info/sct', 'code': '44054006', 'display': 'Type 2 diabetes'}},
    ],
})
version = [p for p in resp.json()['parameter'] if p['name'] == 'return'][0]
concepts = [p for p in resp.json()['parameter'] if p['name'] == 'concept']
print(f'After adding 2 concepts: version={version["valueString"]}, {len(concepts)} concepts')

# Verify subsumption via closure (O(1) lookup)
from medterm4ds.engines.fhir.closure import get_closure_manager
closure = get_closure_manager().get('demo-closure')
print(f'Closure check(73211009, 44054006): {closure.check("73211009", "44054006")}')
print(f'Closure check(44054006, 73211009): {closure.check("44054006", "73211009")}')

---
## Real-World Scenario: Patient Medication Reconciliation

A patient has a FHIR Condition with SNOMED code 44054006 (Type 2 diabetes).
We want to:
1. Get the patient-friendly name
2. Find the canonical ICD-10 code for association lookup
3. Search for related medications
4. Validate that a prescribed code exists

In [ ]:
CONDITION_CODE = '44054006'
CONDITION_SYSTEM = 'http://snomed.info/sct'

# Step 1: Get patient-friendly name + canonical code
resp = client.get('/fhir/CodeSystem/$lookup', params={
    'system': CONDITION_SYSTEM, 'code': CONDITION_CODE,
})
body = resp.json()
display = [p for p in body['parameter'] if p['name'] == 'display'][0]['valueString']
props = {}
for p in body['parameter']:
    if p['name'] == 'property':
        parts = {pt['name']: pt for pt in p['part']}
        props[parts['code']['valueCode']] = parts.get('value', {}).get('valueString', parts.get('value', {}).get('valueCode', '?'))

print(f'Patient-friendly: {props.get("patient-friendly", display)}')
print(f'Canonical ICD-10: {props.get("canonical-code", "?")} ({props.get("canonical-system", "?")})')

# Step 2: Search for related medications
print(f'\nMedications for "{display}":')
show_search('metformin', mode='hybrid', system='http://www.nlm.nih.gov/research/umls/rxnorm')

# Step 3: Validate a prescribed RxNorm code
resp = client.get('/fhir/CodeSystem/$validate-code', params={
    'system': 'http://www.nlm.nih.gov/research/umls/rxnorm', 'code': '860975',
})
result = [p for p in resp.json()['parameter'] if p['name'] == 'result'][0]
print(f'\nPrescribed RxNorm 860975 valid: {result["valueBoolean"]}')

---
## Summary

| Operation | Use case | Latency |
|---|---|---|
| `$lookup` | Display name + properties for a known code | ~5ms |
| `$validate-code` | Check if a code exists | ~5ms |
| `$translate` | Map between code systems (SNOMED ↔ ICD-10) | ~10ms |
| `$subsumes` | Hierarchy relationship check | ~10ms |
| `$expand` filter | EHR autocomplete dropdown | ~50ms |
| `$expand` intensional | All descendants of a concept | ~100ms |
| `$closure` | Pre-compute subsumption for batch checks | ~10ms |
| `$search` lexical | BM25 text search | ~1ms |
| `$search` semantic | Embedding search for novel phrasings | ~100ms |
| `$search` hybrid | Best accuracy (BM25 + re-rank) | ~110ms |

The server is FHIR R4 conformant (validated against HAPI reference server)
and deployable to HF Spaces (0.32 GB lookup DB + pre-computed JSONs).